In [0]:
dbutils.widgets.text("catalog", "claudecatalog", "Catálogo")
dbutils.widgets.text("schema", "supply_chain", "Schema")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

bronze_table = f"{catalog}.{schema}.bronze_orders"
silver_table = f"{catalog}.{schema}.silver_orders"

print(f"Bronze: {bronze_table}")
print(f"Silver: {silver_table}")
df_bronze = spark.table(bronze_table)
df_bronze.printSchema()

In [0]:
import pandas as pd
from pyspark.sql.functions import col, pandas_udf,to_timestamp, count,when,date_diff

In [0]:
df_silver = (
    df_bronze
    # IDs como enteros
    .withColumn("Category_Id", col("Category_Id").cast("int"))
    .withColumn("Customer_Id", col("Customer_Id").cast("int"))
    .withColumn("Department_Id", col("Department_Id").cast("int"))
    .withColumn("Order_Customer_Id", col("Order_Customer_Id").cast("int"))
    .withColumn("Order_Id", col("Order_Id").cast("int"))
    .withColumn("Order_Item_Cardprod_Id", col("Order_Item_Cardprod_Id").cast("int"))
    .withColumn("Order_Item_Id", col("Order_Item_Id").cast("int"))
    .withColumn("Product_Card_Id", col("Product_Card_Id").cast("int"))
    .withColumn("Product_Category_Id", col("Product_Category_Id").cast("int"))
    .withColumn("Customer_Zipcode", col("Customer_Zipcode").cast("int"))
    .withColumn("Order_Zipcode", col("Order_Zipcode").cast("int"))
    # Montos y decimales
    .withColumn("Days_for_shipping_real", col("Days_for_shipping_real").cast("int"))
    .withColumn("Days_for_shipment_scheduled", col("Days_for_shipment_scheduled").cast("int"))
    .withColumn("Benefit_per_order", col("Benefit_per_order").cast("decimal(10,2)"))
    .withColumn("Sales_per_customer", col("Sales_per_customer").cast("decimal(10,2)"))
    .withColumn("Latitude", col("Latitude").cast("double"))
    .withColumn("Longitude", col("Longitude").cast("double"))
    .withColumn("Order_Item_Discount", col("Order_Item_Discount").cast("decimal(10,2)"))
    .withColumn("Order_Item_Discount_Rate", col("Order_Item_Discount_Rate").cast("decimal(5,4)"))
    .withColumn("Order_Item_Product_Price", col("Order_Item_Product_Price").cast("decimal(10,2)"))
    .withColumn("Order_Item_Profit_Ratio", col("Order_Item_Profit_Ratio").cast("decimal(10,4)"))
    .withColumn("Order_Item_Quantity", col("Order_Item_Quantity").cast("int"))
    .withColumn("Sales", col("Sales").cast("decimal(10,2)"))
    .withColumn("Order_Item_Total", col("Order_Item_Total").cast("decimal(10,2)"))
    .withColumn("Order_Profit_Per_Order", col("Order_Profit_Per_Order").cast("decimal(10,2)"))
    .withColumn("Product_Price", col("Product_Price").cast("decimal(10,2)"))
    # Flags / indicadores
    .withColumn("Late_delivery_risk", col("Late_delivery_risk").cast("boolean"))
    .withColumn("Product_Status", col("Product_Status").cast("int"))
    # Fechas (formato del dataset: "1/31/2018 22:56")
    .withColumn("order_date_DateOrders", to_timestamp(col("order_date_DateOrders"), "M/d/yyyy H:mm"))
    .withColumn("shipping_date_DateOrders", to_timestamp(col("shipping_date_DateOrders"), "M/d/yyyy H:mm"))
    # Gobierno de datos: eliminar PII sensible que no debe promoverse
    .drop("Customer_Password")
)

df_silver.printSchema()

In [0]:
cols_to_check = [
    "order_date_DateOrders", "shipping_date_DateOrders",
    "Benefit_per_order", "Sales", "Order_Item_Quantity", "Customer_Id"
]

for c in cols_to_check:
    nulls_bronze = df_bronze.filter(col(c).isNull() | (col(c) == "")).count()
    nulls_silver = df_silver.filter(col(c).isNull()).count()
    print(f"{c}: nulos en Bronze={nulls_bronze} | nulos en Silver={nulls_silver}")

Celda — Deduplicación

In [0]:
# Verificamos si existen duplicados exactos por Order_Item_Id (la granularidad natural del dataset)
total_rows = df_silver.count()
distinct_rows = df_silver.select("Order_Item_Id").distinct().count()

print(f"Total filas: {total_rows}")
print(f"Order_Item_Id distintos: {distinct_rows}")

Celda — Regla de negocio: bandera de entrega tardía consistente

Aquí es donde tu background de logística entra directo al código. El dataset ya trae Late_delivery_risk, pero vamos a calcular una métrica derivada que es estándar en supply chain: si el envío llegó tarde comparando fecha real vs. fecha programada.

Celda — Regla de negocio: bandera de entrega tardía consistente

Aquí es donde tu background de logística entra directo al código. El dataset ya trae Late_delivery_risk, pero vamos a calcular una métrica derivada que es estándar en supply chain: si el envío llegó tarde comparando fecha real vs. fecha programada.

In [0]:
from pyspark.sql.functions import datediff
df_silver = df_silver.withColumn(
    "is_late_delivery",
    when(
        datediff(col("shipping_date_DateOrders"), col("order_date_DateOrders")) > col("Days_for_shipment_scheduled"),
        True
    ).otherwise(False)
)

In [0]:
display(df_silver)

Celda — Escritura de la tabla Silver

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(f"Tabla creada: {silver_table}")
print(f"Total de filas: {spark.table(silver_table).count()}")